In [0]:
file_path = "/Volumes/workspace/default/data_s"

In [0]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

In [0]:
display(df.limit(20).toPandas())

x_Timestamp,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),meter
2019-07-10T00:00:00.000Z,0.021,243.1,1.79,50.02,BR02
2019-07-10T00:03:00.000Z,0.021,242.91,1.8,50.07,BR02
2019-07-10T00:06:00.000Z,0.021,242.46,1.83,50.0,BR02
2019-07-10T00:09:00.000Z,0.02,241.27,1.79,49.95,BR02
2019-07-10T00:12:00.000Z,0.02,240.77,1.79,49.98,BR02
2019-07-10T00:15:00.000Z,0.021,240.97,1.8,49.99,BR02
2019-07-10T00:18:00.000Z,0.02,241.16,1.79,49.99,BR02
2019-07-10T00:21:00.000Z,0.021,241.56,1.79,50.05,BR02
2019-07-10T00:24:00.000Z,0.02,241.64,1.79,50.06,BR02
2019-07-10T00:27:00.000Z,0.02,241.62,1.78,50.08,BR02


In [0]:
df.printSchema()

root
 |-- x_Timestamp: timestamp (nullable = true)
 |-- t_kWh: double (nullable = true)
 |-- z_Avg Voltage (Volt): double (nullable = true)
 |-- z_Avg Current (Amp): double (nullable = true)
 |-- y_Freq (Hz): double (nullable = true)
 |-- meter: string (nullable = true)



In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 2919315
Columns: 6


In [0]:
display(df.describe())

summary,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),meter
count,2919315,2919315,2919315,2919315,2919315
mean,0.014830555113110998,230.35231747858816,1.385987980057119,46.805484115248376,null
stddev,0.020837794195599536,61.83413589489665,1.8519593560613627,12.236474775856886,null
min,0.0,0.0,0.0,0.0,BR02
max,0.288,623.71,126.05,143.82,BR52


In [0]:
df = (
    df
    .withColumnRenamed("X_Timestamp", "timestamp")
    .withColumnRenamed("t_kWh", "energy_kwh")
    .withColumnRenamed("z_Avg Volt", "avg_voltage")
    .withColumnRenamed("z_Avg Curr", "avg_current")
    .withColumnRenamed("y_Freq", "frequency")
    .withColumnRenamed("meter", "meter_id")
)
print(df.columns)

['timestamp', 'energy_kwh', 'z_Avg Voltage (Volt)', 'z_Avg Current (Amp)', 'y_Freq (Hz)', 'meter_id']


In [0]:
from pyspark.sql.functions import countDistinct

df.select(
    countDistinct("meter_id").alias("unique_meters")
).show()

+-------------+
|unique_meters|
+-------------+
|           46|
+-------------+



In [0]:
from pyspark.sql.functions import col, sum, when

missing_values = df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(missing_values)

timestamp,energy_kwh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),meter_id
0,0,0,0,0,0


In [0]:
df = df.dropna(
    subset=[
        "timestamp",
        "energy_kwh",
        "meter_id"
    ]
)

In [0]:
print("Rows after missing-value handling:", df.count())

Rows after missing-value handling: 2919315


In [0]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print("Total Rows:", total_rows)
print("Duplicate Rows:", duplicate_rows)

Total Rows: 2919315
Duplicate Rows: 0


In [0]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- z_Avg Voltage (Volt): double (nullable = true)
 |-- z_Avg Current (Amp): double (nullable = true)
 |-- y_Freq (Hz): double (nullable = true)
 |-- meter_id: string (nullable = true)



In [0]:
df = (
    df
    .withColumn(
        "energy_kwh",
        col("energy_kwh").cast("double")
    )
    .withColumn(
        "z_Avg Voltage (Volt)",
        col("z_Avg Voltage (Volt)").cast("double")
    )
    .withColumn(
        "z_Avg Current (Amp)",
        col("z_Avg Current (Amp)").cast("double")
    )
    .withColumn(
        "y_Freq (Hz)",
        col("y_Freq (Hz)").cast("double")
    )
    .withColumn(
        "meter_id",
        col("meter_id").cast("string")
    )
)

In [0]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "timestamp",
    to_timestamp(
        "timestamp",
        "dd-MM-yyyy HH:mm"
    )
)

In [0]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- z_Avg Voltage (Volt): double (nullable = true)
 |-- z_Avg Current (Amp): double (nullable = true)
 |-- y_Freq (Hz): double (nullable = true)
 |-- meter_id: string (nullable = true)



In [0]:
invalid_timestamp = df.filter(
    col("timestamp").isNull()
).count()

print("Invalid timestamps:", invalid_timestamp)

Invalid timestamps: 0


In [0]:
negative_energy = df.filter(
    col("energy_kwh") < 0
).count()

print("Negative energy readings:", negative_energy)

Negative energy readings: 0


In [0]:
display(
    df.select(
        "z_Avg Voltage (Volt)"
    ).describe()
)

summary,z_Avg Voltage (Volt)
count,2919315
mean,230.35231747858816
stddev,61.83413589489665
min,0.0
max,623.71


In [0]:
print(
    "Invalid voltage readings:",
    df.filter(
        col("z_Avg Voltage (Volt)") <= 0
    ).count()
)

Invalid voltage readings: 186495


In [0]:
display(
    df.select(
        "z_Avg Current (Amp)"
    ).describe()
)

summary,z_Avg Current (Amp)
count,2919315
mean,1.385987980057119
stddev,1.8519593560613627
min,0.0
max,126.05


In [0]:
print(
    "Negative current readings:",
    df.filter(
        col("z_Avg Current (Amp)") < 0
    ).count()
)

Negative current readings: 0


In [0]:
print(
    "Invalid frequency readings:",
    df.filter(
        col("y_Freq (Hz)") <= 0
    ).count()
)

Invalid frequency readings: 186498


In [0]:
numeric_columns = [
    "energy_kwh",
    "z_Avg Voltage (Volt)",
    "z_Avg Current (Amp)",
    "y_Freq (Hz)"
]

for c in numeric_columns:

    q1, q3 = df.approxQuantile(
        c,
        [0.25, 0.75],
        0.01
    )

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    print("\nColumn:", c)
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)


Column: energy_kwh
Q1: 0.002
Q3: 0.018
IQR: 0.016
Lower Bound: -0.022
Upper Bound: 0.041999999999999996

Column: z_Avg Voltage (Volt)
Q1: 235.43
Q3: 255.02
IQR: 19.590000000000003
Lower Bound: 206.04500000000002
Upper Bound: 284.40500000000003

Column: z_Avg Current (Amp)
Q1: 0.2
Q3: 1.74
IQR: 1.54
Lower Bound: -2.11
Upper Bound: 4.05

Column: y_Freq (Hz)
Q1: 49.96
Q3: 50.04
IQR: 0.0799999999999983
Lower Bound: 49.84
Upper Bound: 50.16


In [0]:
q1, q3 = df.approxQuantile(
    "energy_kwh",
    [0.25, 0.75],
    0.01
)

iqr = q3 - q1

lower_energy = q1 - 1.5 * iqr
upper_energy = q3 + 1.5 * iqr

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "energy_outlier",
    when(
        (col("energy_kwh") < lower_energy) |
        (col("energy_kwh") > upper_energy),
        1
    ).otherwise(0)
)


In [0]:
display(
    df.groupBy(
        "energy_outlier"
    ).count()
)

energy_outlier,count
1,218188
0,2701127


In [0]:
from pyspark.sql.functions import (
    to_date,
    hour,
    dayofmonth,
    month,
    year,
    dayofweek
)

In [0]:
df = (
    df
    .withColumn(
        "date",
        to_date("timestamp")
    )
    .withColumn(
        "hour",
        hour("timestamp")
    )
    .withColumn(
        "day",
        dayofmonth("timestamp")
    )
    .withColumn(
        "month",
        month("timestamp")
    )
    .withColumn(
        "year",
        year("timestamp")
    )
    .withColumn(
        "day_of_week",
        dayofweek("timestamp")
    )
)

In [0]:
df = df.withColumn(
    "is_weekend",
    when(
        col("day_of_week").isin(1, 7),
        1
    ).otherwise(0)
)

In [0]:
display(
    df.select(
        "timestamp",
        "date",
        "hour",
        "day",
        "month",
        "year",
        "day_of_week",
        "is_weekend"
    ).limit(20)
)

timestamp,date,hour,day,month,year,day_of_week,is_weekend
2019-07-10T00:00:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:03:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:06:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:09:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:12:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:15:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:18:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:21:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:24:00.000Z,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:27:00.000Z,2019-07-10,0,10,7,2019,4,0


In [0]:
from pyspark.sql.functions import avg, sum

daily_energy = (
    df
    .groupBy(
        "meter_id",
        "date"
    )
    .agg(
        sum("energy_kwh").alias(
            "daily_energy_kwh"
        ),
        avg("z_Avg Voltage (Volt)").alias(
            "avg_voltage"
        ),
        avg("z_Avg Current (Amp)").alias(
            "avg_current"
        ),
        avg("y_Freq (Hz)").alias(
            "avg_frequency"
        )
    )
)

In [0]:
print("Final Rows:", df.count())

Final Rows: 2919315


In [0]:
print("Final Columns:", len(df.columns))

Final Columns: 14


In [0]:
df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- z_Avg Voltage (Volt): double (nullable = true)
 |-- z_Avg Current (Amp): double (nullable = true)
 |-- y_Freq (Hz): double (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- energy_outlier: integer (nullable = false)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)



In [0]:
display(df.limit(20))

timestamp,energy_kwh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),meter_id,energy_outlier,date,hour,day,month,year,day_of_week,is_weekend
2019-07-10T00:00:00.000Z,0.021,243.1,1.79,50.02,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:03:00.000Z,0.021,242.91,1.8,50.07,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:06:00.000Z,0.021,242.46,1.83,50.0,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:09:00.000Z,0.02,241.27,1.79,49.95,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:12:00.000Z,0.02,240.77,1.79,49.98,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:15:00.000Z,0.021,240.97,1.8,49.99,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:18:00.000Z,0.02,241.16,1.79,49.99,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:21:00.000Z,0.021,241.56,1.79,50.05,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:24:00.000Z,0.02,241.64,1.79,50.06,BR02,0,2019-07-10,0,10,7,2019,4,0
2019-07-10T00:27:00.000Z,0.02,241.62,1.78,50.08,BR02,0,2019-07-10,0,10,7,2019,4,0


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/data_s"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/data_s/SM Cleaned Data BR2019.csv,SM Cleaned Data BR2019.csv,142557800,1786123611000


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/sm"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/sm/upi_transactions_2024.csv,upi_transactions_2024.csv,29811789,1786211475000
